# 第97课 · 做得出，更要讲得清——面试（interview）黄金结构与 30 秒 / 3 分钟讲法

**学习目标**
1. 掌握技术面试的「黄金结构」：问题→方法→实现→结果→局限
2. 为 Aurora 的核心模块准备 30 秒 / 3 分钟两种版本的介绍
3. 能回答「为什么不用 librosa / 为什么从零实现」这类追问
4. 模拟面试：录下自己的回答并自评

← **上一课**　[L96 · 白板演练](L96_whiteboard.ipynb)

> 上节课学习了 **白板演练**：DFT / FFT / 注意力机制（attention mechanism）口述推导，面试现场模拟。  
> 本课将探讨 **面试材料整理**。

## 1. 技术面试的黄金结构

研究工程师面试通常包含三类题：

| 题型 | 典型问法 | 评估什么 |
|---|---|---|
| 概念题 | "解释 STFT 的 window size 和 hop size 的 tradeoff" | 是否真懂，还是背下来的 |
| 实现题 | "白板写出 FFT" / "写一个 mel filterbank" | 能否从零推导+实现 |
| 项目题 | "介绍你做的 Aurora 项目" | 能否清晰表达复杂工作 |

**黄金结构（用于项目题）**：
```
① 问题（10 秒）：我在解决什么问题，为什么值得解决
② 方法（30 秒）：我用了什么技术路线，为什么这样选
③ 实现（60 秒）：核心代码/公式是什么，遇到了什么难点
④ 结果（20 秒）：量化结果（quantified results）是什么（数字！），和 baseline 比如何
⑤ 局限（10 秒）：这个方法的局限，下一步怎么改进
```
总共 ~2 分钟。面试官会在任意一步打断追问——你要能展开任何一层。

## 2. Aurora 核心模块的「30 秒版」

每个模块准备两个版本：30 秒电梯版 + 3 分钟详细版。
下面给出模板，✏️ 部分需要你自己填。

In [ ]:
# ✏️ 填写你的 30 秒版模块介绍
# 要求：不超过 3 句话，包含 1 个数字

aurora_30s = {
    "FFT": None,
    # 示例：
    # "FFT": (
    #   "我从零实现了 Cooley-Tukey FFT，时间复杂度 O(N log N)，"
    #   "输出与 numpy.fft 误差在 1e-9 以内。"
    #   "核心是蝶形分治：每层 N/2 个蝶形单元，共 log₂N 层。"
    # ),
    "STFT":    None,
    "Mel":     None,
    "MFCC":    None,
    "CTC":     None,
    "Whisper": None,
    "RAG":     None,
}

empty = [k for k, v in aurora_30s.items() if v is None]
print(f'还有 {len(empty)} 个模块未准备：{empty}' if empty else '全部 30 秒版已就绪 ✅')

## 3. 常见追问与参考回答思路

**Q: 为什么不用 librosa？**
> 不是"librosa 不好"。Aurora 的目标是证明我真正理解每一步——
> 面试官问任何一行代码，我都能推导它。用 librosa 的工程师
> 和能从零写出 Mel filterbank 的研究工程师，展现的是不同层次的理解。

**Q: 你的实现和工业界用的有什么差距？**
> 差距主要在工程层面：我的实现没有 CUDA kernel、没有流式处理、
> 没有经过 production 压测。算法层面是一致的——
> 我的 MFCC 输出和 torchaudio 在相同参数下误差在 1e-6 以内。

**Q: 如果让你把 Aurora 的 ASR 部署到生产，你会怎么做？**
> 先评估延迟需求：实时 < 500ms 要考虑 streaming CTC，
> 离线的话直接用 batch Whisper。然后量化（quantization，INT8）模型，
> 用 TorchScript 或 ONNX 导出，前面加音频预处理流水线（pipeline）。
> Aurora 现在覆盖了特征提取层，推理层下一步补。

**Q: 你如何评估你的模型？**
> ASR 用 WER（Word Error Rate），我在 LibriSpeech test-clean 上
> 跑过 Whisper-small 微调（fine-tuning），WER ~4.2%。特征提取用 cosine similarity
> 和下游分类准确率做代理指标。

## 4. 系统设计题：音频 AI 面试中的高频场景

研究工程师面试常见一类题型：**给你一个业务场景，设计完整的 ML 系统**。
这类题没有唯一答案，考察的是思路的完整性和 tradeoff 意识。

### 经典题目：设计一个实时会议转录系统

面试官通常期待你讨论以下维度：

```
① 需求明确
   - 实时 or 近实时？延迟上限多少？（例：< 2 秒）
   - 单说话人 or 多说话人分离（Diarization）？
   - 目标语言？领域词汇（会议术语、人名）？

② 数据流水线
   麦克风 → 16kHz PCM → 预加重 → 分帧（25ms窗/10ms hop）
   → log-Mel（80维）→ 模型输入

③ 模型选型
   - 实时：Whisper-tiny（39M参数）+ 流式解码（streaming decoding）
   - 高准确率：Whisper-large-v3（1550M）离线批处理
   - 自定义领域：LoRA 微调 Whisper-small

④ 延迟拆解
   音频缓冲（500ms）+ 特征提取（~5ms）+ 推理（~100ms）
   → 总延迟约 600ms，满足 <2s 要求

⑤ 评估指标
   WER（Word Error Rate）、RTF（Real-Time Factor < 1.0）
   、延迟 P95

⑥ 可能的改进
   CTC 流式解码（不等整句结束）、Speaker Diarization、
   自定义词汇热词注入
```

**面试技巧**：先快速过一遍所有维度，再让面试官选一个深入。
不要一上来就钻到细节——没有整体框架是面试最常见的失误。

## 5. 编程题准备：音频 AI 的高频手写题

除了白板推导（L96），面试也会要求你在 IDE 里当场写代码。
以下是音频 AI 岗位最常见的几类编程题：

| 题目类型 | 例子 | 对应 Aurora 课程 |
|---|---|---|
| 信号生成 | 写出 `generate_sine(freq, duration, sr)` | L33 |
| 手写 DFT | O(N²) 的朴素 DFT 矩阵乘法 | L37 |
| 帧切分 | `frame_signal(x, win_len, hop)` | L44 |
| Mel 映射 | `hz_to_mel(f)` 和 `mel_to_hz(m)` | L46 |
| 余弦相似度（cosine similarity） | `cosine_sim(a, b)` | L80 |
| WER 计算 | 用编辑距离（edit distance）计算 Word Error Rate | L67 |

In [ ]:
# ✏️ 当场限时练习：不看任何参考，5 分钟内写出以下两个函数

import numpy as np

def hz_to_mel(f):
    """把线性频率 Hz 转换为 Mel 标度。"""
    raise NotImplementedError("实现 HTK mel 公式：2595 * log10(1 + f/700)")

def frame_signal(x, win_len, hop):
    """
    把 1D 信号 x 切成帧。
    返回 shape (n_frames, win_len) 的 2D 数组。
    """
    raise NotImplementedError("用切片或 np.lib.stride_tricks 实现帧切分")

# 自动检查
f_test = np.array([0, 1000, 4000])
mel_ref = 2595 * np.log10(1 + f_test / 700)
assert np.allclose(hz_to_mel(f_test), mel_ref, atol=0.1), "hz_to_mel 公式有误"
print("hz_to_mel ✅")

x = np.arange(20, dtype=float)
frames = frame_signal(x, win_len=8, hop=4)
assert frames.shape == (4, 8), f"期望 (4,8)，得到 {frames.shape}"
print("frame_signal ✅")

## 6. 行为面试：STAR 结构

研究工程师岗位也会问行为题（Behavioral Questions），
用 STAR 结构回答能让回答有血有肉：

| 字母 | 含义 | 说什么 |
|---|---|---|
| **S**ituation | 背景 | 当时在什么项目/时间点 |
| **T**ask | 任务 | 你的具体职责是什么 |
| **A**ction | 行动 | 你做了哪些具体步骤（用"我"而非"我们"） |
| **R**esult | 结果 | 量化结果，或者学到了什么 |

**Aurora 场景下的常见行为题：**

- "讲一个你遇到的技术难点，你是怎么解决的"
  → 可以用 FFT 实现时的数值稳定性问题，或者 CTC 前向算法的边界条件
- "讲一个你做的实验没有达到预期的经历"
  → 可以用 Whisper 微调后 WER 没有改善，分析原因
- "你如何保证代码质量"
  → Aurora 的每个函数都有 assert 测试对齐 numpy 参考实现

## 7. 面试结束时：你要问的问题

面试官说"你有什么问题吗"——这是机会，不是客套。
一个好问题能让面试官看出你是真的在认真考虑这份工作，而不只是来找 offer。

**好问题示例：**

- "您团队目前的技术栈里，有哪块是你们希望在接下来一年显著改进的？"
- "您加入这个团队后，觉得最超出预期的挑战是什么？"
- "刚入职的前 3 个月，最重要的交付物通常是什么？"
- "团队的研究和工程比例大概是怎么分配的？"

**不好的问题：**
- "薪资是多少？"（留到 HR 环节）
- "你们有多少假期？"（重点偏了）
- "没什么问题了"（错失机会）

## 练习：模拟面试

用手机录音 2 分钟，回答下面这道题，然后按评分标准自评。

**题目**：请介绍你的 Aurora 项目，重点说一说你是如何实现 FFT 的，
以及为什么要从零实现而不是调用 numpy.fft。

In [ ]:
# 自评表（录完再填）
mock_interview = {
    "duration_seconds":     None,   # 你说了多少秒？
    "mentioned_numbers":    None,   # 说了哪些数字/结果？True/False
    "explained_why_scratch": None,  # 讲清楚了为什么从零写？True/False
    "covered_limitation":   None,   # 提到局限了吗？True/False
    "filler_words_count":   None,   # 数一下「那个/嗯/就是」出现几次
    "next_improvement":     None,   # 下次要改进什么？
}

score = sum(1 for k in ["mentioned_numbers","explained_why_scratch","covered_limitation"]
            if mock_interview.get(k) is True)
if all(v is not None for v in mock_interview.values()):
    print(f'关键项得分：{score}/3')
    if score == 3:
        print('✅ 核心要素齐全，继续打磨流畅度')
    else:
        print('⚠️  有要素缺失，回看黄金结构后再录一遍')
else:
    print('先完成录音自评，再运行这格')

## 8. 3 分钟版：以 FFT 为例

30 秒版让面试官知道你做了什么，3 分钟版让他们相信你真的懂。
以 FFT 为例，3 分钟版的结构：

**（0:00-0:20）背景**
"FFT 是整个 Aurora 音频处理链路的核心——STFT、Mel、MFCC 全部依赖它。
我选择从零实现而不是调用 numpy.fft，是为了能在白板上推导每一步。"

**（0:20-1:20）方法**
"Cooley-Tukey 分治：把 N 点 DFT 分成奇偶两个 N/2 点 DFT，递归到 N=1 终止。
每层有 N/2 个蝶形单元，每个单元做一次复数乘法（旋转因子）加两次复数加法。
log₂N 层，所以总复杂度 O(N log N)。"

**（1:20-1:50）实现与验证**
"核心是 `butterfly(E, O, k, N)` 函数，输入两个 N/2 点复数 DFT，
输出 `E + W·O` 和 `E - W·O`，其中 W = e^{-2πik/N}。
用 `np.allclose(my_fft(x), np.fft.fft(x), atol=1e-9)` 验证，误差在 1e-9 以内。"

**（1:50-2:10）局限**
"我的实现是纯 Python，没有 SIMD/CUDA 优化，
在 N=4096 时比 numpy.fft 慢约 30 倍。工业场景应该用 FFTW 或 cuFFT。"

**（2:10-3:00）回答追问**
留出时间让面试官问"那为什么不直接用 numpy.fft"——
你已经在第一句话里埋下了答案，这时候展开就好。

## 9. 远程面试的额外准备

现在大多数初轮面试是线上进行的，技术问题之外还有环境问题。

### 技术设置

- **摄像头**：眼神看镜头，不是看屏幕——提前练习这个习惯
- **麦克风**：用耳机麦克风，避免回声；面试前用 `audacity` 或系统录音检查音质
- **网络**：有线优于无线；提前测速，确保上行 > 5 Mbps
- **共享屏幕**：关掉通知（macOS: 勿扰模式），提前打开你要展示的文件/终端

### 编程环境

面试前准备好一个干净的 IDE 窗口：
- 字体调大（面试官看你的屏幕，字要清晰）
- 关掉 Copilot 等 AI 补全（面试官要看你自己的思路）
- 准备好可以直接运行 Python 的终端

### 面试中出现卡顿的处理

如果网络卡顿或听不清，直接说"不好意思，网络有点问题，你能重复一下吗？"
比装作听懂了然后答错要好得多。

## 10. ✏️ 准备 3 个完整的 STAR 故事

在面试前写出来，不是背，而是**确认故事的骨架是完整的**。
写出来之后你会发现，很多"我以为我能讲清楚"的故事其实缺少量化结果。

In [ ]:
# ✏️ 写出 3 个 STAR 故事框架（不限长度）
star_stories = {
    "Story 1: 技术难点": {
        "situation":  None,   # 什么项目，什么时间点
        "task":       None,   # 你的职责是什么
        "action":     None,   # 你做了哪些具体步骤（我 而非 我们）
        "result":     None,   # 量化结果或学到了什么
    },
    "Story 2: 失败/调整经历": {
        "situation":  None,
        "task":       None,
        "action":     None,
        "result":     None,
    },
    "Story 3: 主动改进某件事": {
        "situation":  None,
        "task":       None,
        "action":     None,
        "result":     None,
    },
}

for story_name, parts in star_stories.items():
    filled = sum(1 for v in parts.values() if v is not None)
    complete = "✅" if filled == 4 else f"⬜ {filled}/4"
    print(f"{complete}  {story_name}")
    if filled < 4:
        missing = [k for k, v in parts.items() if v is None]
        print(f"       缺少：{missing}")

## 11. 不同公司的面试侧重

研究工程师的面试形式因公司类型而异：

| 公司类型 | 典型轮次 | 侧重 |
|---|---|---|
| 大厂语音组（Google、Meta） | 5-6 轮：算法+系统设计+行为+论文讨论 | 理论深度 + 规模化工程 |
| 音频 AI 初创 | 2-4 轮：take-home + 技术面 + 文化面 | 能否快速上手交付 |
| 高校实验室 | 1-2 轮：论文讨论 + 研究方向聊天 | 研究思维 + 发表潜力 |
| 外包/服务公司 | 1-2 轮：实际项目场景题 | 解决真实问题的工程能力 |

**Aurora 最对应的面试类型**：音频 AI 初创（take-home 展示 Aurora 代码）
和大厂技术面（白板推导 FFT/MFCC）。
大厂系统设计轮可能问"设计一个语音助手的完整后端"——用 L92 的流水线框架回答。

## 本课收束

面试是一个可以练习的技能，不只是知识的展示。
每一模块做到能在 2 分钟内讲清楚，就已经超过大多数候选人。

**下一课 L98**：课程总结——6 个月做了什么，还差什么，Aurora v2 的方向。

---

→ **下一课**　[L98 · 复盘](L98_retrospective.ipynb)

> 下节课将学习 **复盘**：6 个月里程碑回顾，方法论总结，成长曲线量化。